
# A blinking acceptor, and why intensity alone cannot see it

This is the photophysically realistic case. Three conformational states with
**Gaussian distance distributions**, a **multi-exponential donor** quenched
**homogeneously** by transfer, and an **acceptor that blinks** on a 10 µs
timescale — faster than a burst, so blinking happens *within* the observation.

The difficulty is specific. A dark acceptor produces no acceptor photons, so it
reports $E = 0$; a large donor–acceptor distance reports a small
$E$. **By intensity the two are the same observation.** They differ only
in the donor lifetime — a dark acceptor leaves the donor completely unquenched,
while a distant acceptor still drains it — and that is a micro-time signal.

The state space is a product: *conformation* × *acceptor photophysics*, the same
construction the single-photon smFRET literature uses for superstates. While the
acceptor is dark the conformation is unobservable, so it collapses to one dark
state rather than three indistinguishable ones — three identical emissions would
be unidentifiable, and no amount of data would separate them.

<div class="alert alert-info"><h4>Note</h4><p>**Homogeneous quenching** means every donor component sees the same transfer
   rate at a given distance: $1/\tau_{i,DA} = 1/\tau_i + k_{FRET}(R)$.
   The donor's multi-exponentiality is intrinsic — local environment — and FRET
   adds one rate to all of it. With a distance *distribution* on top, a state's
   donor decay is a double sum over components and distances: 30 exponentials
   from 2 components and 15 quadrature nodes. That is why the emission is
   generated from parameters rather than fitted as a table.</p></div>


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import tttrlib

STATE_COLORS = ["#4e79a7", "#59a14f", "#e15759", "#b07aa1"]

# --- photophysics ----------------------------------------------------------
R0 = 52.0
A_D = np.array([0.6, 0.4])            # unquenched donor amplitudes
TAU_D = np.array([3.8, 1.4])          # unquenched donor lifetimes, ns
TAU_REF = float((A_D * TAU_D).sum() / A_D.sum())   # species average; defines R0
SIGMA_R = 6.0                         # linker width, Å
N_NODE = 15
TAU_A = 2.2                           # acceptor lifetime, ns
R_STATES = (40.0, 52.0, 65.0)

N_BINS, SPAN_NS = 64, 16.0
DT_NS = SPAN_NS / N_BINS

# Macro-time tick = 1 µs, so a 10 µs blink is 0.1 per tick.
K_BLINK_OFF, K_BLINK_ON = 0.10, 0.10   # 10 µs dark, 10 µs bright
K_CONF = 2e-3                          # conformational exchange, ~0.5 ms


def donor_spectrum(r_mean):
    """(amplitudes, lifetimes, E) for one distance distribution."""
    r = np.linspace(max(r_mean - 3 * SIGMA_R, 15.0), r_mean + 3 * SIGMA_R, N_NODE)
    w = np.exp(-0.5 * ((r - r_mean) / SIGMA_R) ** 2)
    w /= w.sum()
    kf = (1.0 / TAU_REF) * (R0 / r) ** 6
    tau_da = 1.0 / (1.0 / TAU_D[:, None] + kf[None, :])
    e_ic = 1.0 - tau_da / TAU_D[:, None]
    amp = (A_D[:, None] / A_D.sum()) * w[None, :] * (1.0 - e_ic)
    e = float(((A_D[:, None] / A_D.sum()) * w[None, :] * e_ic).sum())
    a, t = amp.ravel(), tau_da.ravel()
    keep = (a > 1e-6) & (t > 1e-3)
    return a[keep] / a[keep].sum(), t[keep], e


def build_spec(distances):
    """Four states: three bright conformations plus one dark-acceptor state."""
    spec = tttrlib.HmmEmissionSpec.uniform(4, 2, N_BINS, DT_NS, TAU_REF)
    for k, r in enumerate(distances):
        amp, tau, e = donor_spectrum(r)
        spec.set_stream_probability(k, 0, 1.0 - e)
        spec.set_stream_probability(k, 1, e)
        spec.set_spectrum(k, 0, tttrlib.HmmLifetimeSpectrum(
            tttrlib.VectorDouble(list(amp)), tttrlib.VectorDouble(list(tau))))
        spec.set_spectrum(k, 1, tttrlib.HmmLifetimeSpectrum(TAU_A))
    # Dark acceptor: no transfer, so the donor keeps its full multi-exponential
    # decay and no acceptor photons appear.
    spec.set_stream_probability(3, 0, 1.0)
    spec.set_stream_probability(3, 1, 0.0)
    spec.set_spectrum(3, 0, tttrlib.HmmLifetimeSpectrum(
        tttrlib.VectorDouble(list(A_D / A_D.sum())),
        tttrlib.VectorDouble(list(TAU_D))))
    spec.set_spectrum(3, 1, tttrlib.HmmLifetimeSpectrum(TAU_A))
    return spec


def transition_matrix():
    """Conformation × photophysics, with the dark state shared."""
    a = np.zeros((4, 4))
    for i in range(3):
        for j in range(3):
            if i != j:
                a[i, j] = K_CONF
        a[i, 3] = K_BLINK_OFF                 # acceptor goes dark
        a[3, i] = K_BLINK_ON / 3.0            # lights up in some conformation
    np.fill_diagonal(a, 0.0)
    np.fill_diagonal(a, 1.0 - a.sum(axis=1))
    return a

## Simulate
Blinking is fast relative to the photon rate, so the acceptor toggles several
times inside a single burst.



In [ ]:
A_TRUE = transition_matrix()
spec_true = build_spec(R_STATES)
true = tttrlib.HmmModel([0.25] * 4, list(A_TRUE.ravel()), spec_true.build())
true.n_micro_bins = N_BINS

E_TRUE = spec_true.build_np().sum(axis=2)[:, 1]
TAU_TRUE = [float((spec_true.build_np()[k, 0] / spec_true.build_np()[k, 0].sum()
                   * np.arange(N_BINS) * DT_NS).sum()) for k in range(4)]
print("state      E      mean donor micro-time (ns)")
for k, name in enumerate(["R=40", "R=52", "R=65", "dark"]):
    print(f"{name:>6} {E_TRUE[k]:>7.3f} {TAU_TRUE[k]:>22.2f}")

rng = np.random.default_rng(4)
obs, powers = true.obs_np, {}
times, symbols, truth = [], [], []
for _ in range(80):
    t = np.cumsum(rng.integers(1, 8, size=500)).astype(np.int64)   # ~4 µs spacing
    state = rng.integers(0, 4)
    row, st = [], []
    for i in range(len(t)):
        if i:
            dt = int(t[i] - t[i - 1])
            if dt not in powers:
                powers[dt] = np.linalg.matrix_power(A_TRUE, dt)
            state = rng.choice(4, p=powers[dt][state])
        st.append(state)
        row.append(int(rng.choice(obs.shape[1], p=obs[state])))
    times.append(t.tolist())
    symbols.append(row)
    truth.append(st)
truth = np.concatenate(truth)

streams = [[y // N_BINS for y in b] for b in symbols]
bins = [[y % N_BINS for y in b] for b in symbols]

## Refine the distances
The parameterised M-step re-fits a *mono-exponential* lifetime, and these
spectra hold thirty components each — so it cannot move them. The distances
are refined instead through :func:`HMM.evaluate`, which hands back one
E-step's sufficient statistics and lets the physics own the M-step. Exactly
the loop in ``plot_hmm_distance_refinement.py``, with a fourth state whose
decay is *known* rather than fitted.



In [ ]:
eng = tttrlib.HMM()
eng.set_bursts_micro(times, streams, bins, 2, N_BINS, DT_NS)
print(f"\n{eng.get_n_photons()} photons, {eng.get_n_symbols()} symbols, 4 states")


def q_of_distance(r, state, gamma_obs):
    """Q for one bright state as a function of its distance."""
    amp, tau, e = donor_spectrum(r)
    one = tttrlib.HmmEmissionSpec.uniform(1, 2, N_BINS, DT_NS, TAU_REF)
    one.set_stream_probability(0, 0, 1.0 - e)
    one.set_stream_probability(0, 1, e)
    one.set_spectrum(0, 0, tttrlib.HmmLifetimeSpectrum(
        tttrlib.VectorDouble(list(amp)), tttrlib.VectorDouble(list(tau))))
    one.set_spectrum(0, 1, tttrlib.HmmLifetimeSpectrum(TAU_A))
    table = np.asarray(one.build_np())[0]
    return float((gamma_obs[state].reshape(2, N_BINS)
                  * np.log(np.maximum(table, 1e-300))).sum())


def refine(distances, gamma_obs, lo=28.0, hi=85.0):
    g = (np.sqrt(5) - 1) / 2
    out = []
    for state in range(len(distances)):
        a, b = lo, hi
        c, d = b - g * (b - a), a + g * (b - a)
        fc, fd = q_of_distance(c, state, gamma_obs), q_of_distance(d, state, gamma_obs)
        for _ in range(40):
            if b - a < 1e-3:
                break
            if fc > fd:
                b, d, fd = d, c, fc
                c = b - g * (b - a); fc = q_of_distance(c, state, gamma_obs)
            else:
                a, c, fc = c, d, fd
                d = a + g * (b - a); fd = q_of_distance(d, state, gamma_obs)
        out.append(0.5 * (a + b))
    return np.array(out)


R = np.array([45.0, 50.0, 58.0])                    # deliberately clustered
A = transition_matrix()
for _ in range(15):
    model = tttrlib.HmmModel([0.25] * 4, list(A.ravel()), build_spec(R).build())
    model.n_micro_bins = N_BINS
    ev = eng.evaluate(model)
    xi = ev.xi_np
    A = xi / xi.sum(1, keepdims=True)
    R = refine(R, ev.gamma_obs_np(4))

fit = tttrlib.HmmModel([0.25] * 4, list(A.ravel()), build_spec(R).build())
fit.n_micro_bins = N_BINS
print(f"distances  {R.round(1)}   truth {R_STATES}")
print(f"blink off  {A[0, 3]:.4f} / tick   truth {K_BLINK_OFF}")

decoded = np.asarray(eng.viterbi_path(fit)[0])
dark_true = truth == 3
dark_found = decoded == 3
recall = float(np.mean(dark_found[dark_true]))
precision = float(np.mean(dark_true[dark_found])) if dark_found.any() else 0.0
print(f"dark state: recall {recall:.3f}, precision {precision:.3f}")

## Plot
Left: the donor decay of every state. The dark state is the slowest — nothing
is draining it — and the $E$ in each label is what an intensity-only
analysis would see. R = 65 Å and the dark acceptor are 0.22 against 0.00 in
$E$, a difference a burst histogram smears out, and 2.26 against 3.00 ns
in lifetime, which is per-photon. Right: what the refinement recovered.



In [ ]:
fig, (ax_decay, ax_fit) = plt.subplots(1, 2, figsize=(10.5, 4.0))

t_axis = np.arange(N_BINS) * DT_NS
tab = spec_true.build_np()
labels = [f"R = {r:.0f} Å,  E = {E_TRUE[k]:.2f}" for k, r in enumerate(R_STATES)]
labels.append(f"dark acceptor,  E = {E_TRUE[3]:.2f}")
for k, name in enumerate(labels):
    d = tab[k, 0] / tab[k, 0].sum()
    ax_decay.plot(t_axis, d, lw=2, color=STATE_COLORS[k],
                  ls="--" if k == 3 else "-", label=name)
ax_decay.set_yscale("log")
ax_decay.set_ylim(1e-4, 1)
ax_decay.set_xlabel("micro-time (ns)")
ax_decay.set_ylabel("P(bin | state, donor)")
ax_decay.set_title("Donor decay per state\nhomogeneous quenching, multi-exp donor",
                   fontsize=10, loc="left")
ax_decay.legend(frameon=False, fontsize=9)
ax_decay.grid(alpha=0.25, lw=0.6)

x = np.arange(3)
ax_fit.bar(x - 0.18, R_STATES, width=0.34, color="#9aa4ad", label="truth")
ax_fit.bar(x + 0.18, R, width=0.34, color=STATE_COLORS[:3], label="refined")
for i, r in enumerate(R):
    ax_fit.text(i + 0.18, r + 1.0, f"{r:.1f}", ha="center", fontsize=9)
ax_fit.set_xticks(x)
ax_fit.set_xticklabels([f"state {i}" for i in range(3)])
ax_fit.set_ylabel("mean donor–acceptor distance (Å)")
ax_fit.set_ylim(0, 78)
ax_fit.set_title(
    f"Refined from a clustered start (45/50/58 Å)\n"
    f"blinking {A[0, 3]:.3f}/tick vs {K_BLINK_OFF}; "
    f"dark recall {recall:.2f}", fontsize=10, loc="left")
ax_fit.legend(frameon=False, fontsize=9)
ax_fit.grid(alpha=0.25, lw=0.6, axis="y")

fig.tight_layout()
plt.show()

## What this shows
The blinking is recovered almost exactly — 0.099 per tick against 0.100 — and
the two closer distances to within about 1.5 Å. **The 65 Å state comes back at
61 Å, and that error is the physics, not the fit.** At 65 Å the transfer is
weak, so the state's donor decay sits close to the unquenched one and its
separation from the dark state rests on a small lifetime difference. Pulling
the distance in slightly buys donor photons that the dark state would
otherwise have to explain. Precision on the dark state (0.72) is lower than
recall (0.91) for the same reason: the confusion runs one way, dark photons
absorbed from the low-FRET state.

This is the identifiability limit of the measurement rather than of the
method, and it is worth stating plainly because it is generic: **a dark
acceptor and a very distant one converge as** $R$ **grows**, and no
estimator can separate them once the quenching falls below the lifetime
resolution. Reporting a distance out here needs the interval that
:func:`HMM.sample` gives, not a point estimate.

Note what is *not* fitted. Each state's decay is thirty exponentials, from two
donor components crossed with fifteen distance nodes, and all of it follows
from one scalar — the mean distance — through homogeneous quenching. A free
emission table would carry 128 numbers per state and could place a dark
acceptor anywhere; the parameterisation makes that answer unreachable.

